# Libraries

Import libraries needed in this project.

In [1]:
import warnings
warnings.filterwarnings('ignore')
import pywt
import cv2  #For image and video processing and visualization
import os   #To interact with operating system and files
import numpy as np  #For matrix operations
import random   #To generate random numbers
from sklearn.mixture import GaussianMixture  #For clustering
from sklearn.cluster import DBSCAN   #For clustering
from sklearn.cluster import MeanShift, estimate_bandwidth  #For clustering
#import numba
from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage.metrics import structural_similarity as ssim
from scipy.stats import entropy
import torch
import torch.nn.functional as Fun
from torchvision import transforms
from torch import optim
import time
from torch import nn
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader
from scipy.interpolate import splprep, splev
from scipy.ndimage import gaussian_filter
from datetime import timedelta
import datetime
import seaborn as sns
from skimage.color import rgb2gray
from torch.optim.lr_scheduler import StepLR
import torchvision
import kornia.color as kcolor
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import math
import pytorch_msssim
import torchvision.models as models
from scipy.ndimage import uniform_filter
import pandas as pd
from scipy.ndimage import gaussian_filter
from scipy.stats import spearmanr
from skimage.draw import bezier_curve
from torchvision.models.optical_flow import raft_large, Raft_Large_Weights
from torchvision.models.optical_flow import raft_small, Raft_Small_Weights
from torchvision.utils import flow_to_image
from pytorch_msssim import ssim as t_ssim

# Import Own Libraries

In [2]:
import import_ipynb
from Video_and_Display_Functions import *
from OpticalFlow_Functions import *
from Inconsistencies_Functions import *
from Cartoon_Functions import *
from Quality_Consistency_Functions import *
from Features_Functions import *

# Set GPU or CPU

It automatically sets gpu or cpu usage for pytorch.

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


# Torch to Numpy and Vice versa

In [4]:
def Frame2Torch(Frame, Size=None, Normalize=False, Denormalize=False):
    if Size is not None:
        interpolation = cv2.INTER_CUBIC if Size[0] > Frame.shape[1] or Size[1] > Frame.shape[0] else cv2.INTER_AREA
        Frame = cv2.resize(Frame, Size, interpolation=interpolation)
    if Normalize:
        Frame = Frame.astype(float) / (Frame.max()+1e-12)
    if Denormalize:
        Frame = Frame.astype(float) * 255.0
    tensor = torch.from_numpy(Frame)
    if len(Frame.shape) == 3:
        return tensor.permute(2, 0, 1).unsqueeze(0)
    else:
        return tensor

In [5]:
def Frame2Numpy(Frame, Size=None, Normalize=False, Denormalize=False):
    if Frame.ndim == 2:  # If the tensor has shape (N, M)
        Frame = Frame.detach().cpu().numpy()
    elif Frame.ndim == 3:  # If the tensor has shape (C, H, W)
        Frame = Frame.squeeze(0).permute(1, 2, 0).detach().cpu().numpy()
    if Size is not None:
        interpolation = cv2.INTER_CUBIC if Size[0] > Frame.shape[0] or Size[1] > Frame.shape[1] else cv2.INTER_AREA
        Frame = cv2.resize(Frame, Size, interpolation=interpolation)
    if Normalize:
        Frame = Frame.astype(float) / 255.0
    if Denormalize:
        Frame = (Frame.astype(float) * 255.0).clip(0, 255).astype(np.uint8)
    return Frame

In [6]:
def OF2Numpy(OF):
    return OF.permute(1,2,0).cpu().numpy()

# Sigmoid Function

In [67]:
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

In [78]:
def similarity(A, B, metric="L1", sigma=0.5, eps=1e-10, thresh=None):
    thresh = np.clip(np.mean(A)+1.2*np.std(A),0.1,0.5) if thresh is None else 0.5

    if metric == "L1" or metric == "absdiff":
        sim = 1.0 - np.abs(A - B)
    elif metric == "L2":
        sim = 1 - (A - B)**2
    elif metric == "ProbSim":
        sim = (1 - A)*(1 - B) + A*B
    elif metric == "Dice":
        sim = 2 * A * B / (A**2 + B**2 + eps)
    elif metric == "Bhat":
        sim = np.sqrt(A * B) + np.sqrt((1 - A)*(1 - B))
    elif metric == "Hel":
        sim = 1 - np.sqrt(0.5 * ((np.sqrt(A) - np.sqrt(B))**2 + (np.sqrt(1 - A) - np.sqrt(1 - B))**2))
    elif metric == "Tani":
        sim = (A * B) / (A**2 + B**2 - A * B + eps)
    elif metric == "KL":
        KL_AB = A * np.log((A + eps) / (B + eps)) + (1 - A)*np.log((1 - A + eps)/(1 - B + eps))
        KL_BA = B * np.log((B + eps) / (A + eps)) + (1 - B)*np.log((1 - B + eps)/(1 - A + eps))
        sim = 1 / (1 + KL_AB + KL_BA)
    elif metric == "MutualInfo":
        num = A*B+(1-A)*(1-B)
        den = A*(1-B)+B*(1-A)+eps
        f = np.log(1+num/den)
        sim =  f/(1+f)
    elif metric == "RBF":
        sim = np.exp(-((A - B)**2) / (2 * sigma**2))
    elif metric == "SSIM":
        sim = (2 * A * B + eps) / (A**2 + B**2 + eps)
    elif metric == "PSNR":
        psnr = 10 * np.log10(1 / ((A - B)**2 + eps))
        psnr_norm = (psnr - np.min(psnr)) / (np.max(psnr) - np.min(psnr) + eps)
        sim = psnr_norm
    elif metric == "Pearson":
        pearson = (A - np.mean(A)) * (B - np.mean(B))
        sim = (pearson - np.min(pearson)) / (np.max(pearson) - np.min(pearson) + eps)
    elif metric == "Canberra":
        sim = 1 - np.abs(A - B) / (np.abs(A) + np.abs(B) + eps)
    elif metric == "Log":
        log_loss = -(B * np.log(A + eps) + (1 - B) * np.log(1 - A + eps))
        logloss_sim = np.exp(-log_loss)
        sim = (logloss_sim - np.min(logloss_sim)) / (np.max(logloss_sim) - np.min(logloss_sim)+1e-10)
    elif metric == "Accuracy":
        TP = np.sum((A >= 0.5) & (B >= 0.5))
        TN = np.sum((A < 0.5) & (B < 0.5))
        FP = np.sum((A >= 0.5) & (B < 0.5))
        FN = np.sum((A < 0.5) & (B >= 0.5))
        N_Total = TP + TN + FP + FN
        sim = (TP + TN) / N_Total
    elif metric == "Precision":
        TP = np.sum((A >= 0.5) & (B >= 0.5))
        TN = np.sum((A < 0.5) & (B < 0.5))
        FP = np.sum((A >= 0.5) & (B < 0.5))
        FN = np.sum((A < 0.5) & (B >= 0.5))
        N_Total = TP + TN + FP + FN
        den = (TP + FP + eps)
        sim = TP / den
    elif metric == "Recall":
        TP = np.sum((A >= 0.5) & (B >= 0.5))
        TN = np.sum((A < 0.5) & (B < 0.5))
        FP = np.sum((A >= 0.5) & (B < 0.5))
        FN = np.sum((A < 0.5) & (B >= 0.5))
        N_Total = TP + TN + FP + FN
        den = (TP + FN + eps)
        sim = TP / den
    elif metric == "F1":
        TP = np.sum((A >= 0.5) & (B >= 0.5))
        TN = np.sum((A < 0.5) & (B < 0.5))
        FP = np.sum((A >= 0.5) & (B < 0.5))
        FN = np.sum((A < 0.5) & (B >= 0.5))
        N_Total = TP + TN + FP + FN
        Precision = TP / (TP + FP + eps)
        Recall = TP / (TP + FN + eps)
        sim = 2 * (Precision * Recall) / (Precision + Recall + eps)
    else:
        raise ValueError(f"Unknown metric '{metric}'")
        return -0
    sim_value = np.clip(np.mean(sim), 0, 1)
    return sim_value

# Fast Normalization

In [79]:
def fast_norm(data):
    arr = np.array(data)
    spatial_axes = (1, 2) 
    amin = arr.min(axis=spatial_axes, keepdims=True)
    amax = arr.max(axis=spatial_axes, keepdims=True)
    dist = amax - amin
    return np.divide(arr - amin, dist, out=np.zeros_like(arr), where=dist != 0)